# semantic_search/01 — Aggregate note embeddings into 3x768 patient features

Mean-pools each patient's progress, imaging, and pathology notes separately, then concatenates
the three blocks into one 2,304-feature vector. No alternative representation is produced.

**Runs after** `1_data/01_preprocessing` and `1_data/03_prediction_datasets` (needs the knitted
embedding metadata + array, and `cohort_df` for the treatment anchor). **Runs before**
`semantic_search/02_cluster`. Nothing downstream of the manuscript pipeline depends on it — this
arm is exploratory and additive.

## One 3x768 feature space, two optional windows

| Window | Notes included |
|---|---|
| `alltime` | every note the patient has, no anchor |
| `pretreatment` | notes strictly before `first_treatment_date` |

Each note type is pooled with a plain unweighted mean. Patients missing any of the three finite
768-dimensional blocks are complete-cased out before the blocks are concatenated.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<24} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a semantic_search stage as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={time.time() - t0:,.1f}s", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

`SPACES` is fixed to `['concat']` by `semantic_search.common`. Select the note windows here.
The stage is skip-if-exists; set `OVERWRITE` to rebuild the concatenated artifacts.

In [ ]:
MODULE = "semantic_search.aggregate_embeddings"

WINDOWS = common.DEFAULT_WINDOWS  # ["alltime"]; add "pretreatment" only if wanted
OVERWRITE = False                 # True -> rebuild feature files that already exist
LIMIT_MRNS = None                 # int -> debug run on the first N patients only

print(f"windows:   {WINDOWS}")
print(f"spaces:    {common.SPACES}")
print(f"overwrite: {OVERWRITE}")
if LIMIT_MRNS:
    print(f"LIMIT_MRNS={LIMIT_MRNS} -- DEBUG RUN, outputs are not the real cohort")

## Preconditions

The embedding array is the large input. This cell does not raise.

In [ ]:
check_inputs([
    ("note embeddings meta",  os.path.join(config.NOTES_PATH,
                                           "full_clinical_notes_embeddings_metadata.parquet")),
    ("note embeddings array", os.path.join(config.NOTES_PATH,
                                           "full_clinical_notes_embeddings_as_array.npy.zst")),
    ("cohort",                os.path.join(config.SURV_PATH, "cohort_df.parquet")),
])

try:
    import zstandard  # noqa: F401
    print("[ok ] zstandard importable (needed to decompress the embedding array)")
except ImportError:
    print("[MISSING] zstandard not importable - run this on the cluster kernel.")

## Pre-flight: what already exists

Read-only census, so a resumed run shows what it will skip.

In [ ]:
existing = common.available_pairs()
print(f"{len(existing)} / {len(common.SPACES) * len(common.WINDOWS)} feature files present")
for space in common.SPACES:
    marks = " ".join(
        f"{w}={'yes' if (space, w) in existing else 'no '}" for w in common.WINDOWS)
    print(f"  {space:10s} {marks}")

## Run

In [ ]:
args = ["--windows", *WINDOWS]
if OVERWRITE:
    args.append("--overwrite")
if LIMIT_MRNS:
    args += ["--limit-mrns", str(LIMIT_MRNS)]

rc = run_module(MODULE, args)
if rc != 0:
    print("\nStage failed - see the traceback above.")

## Summary

`n_patients` is the three-note-type complete-case cohort size. `n_features` should be 2,304 for
the current encoder. Note volume remains a documentation-intensity check.

In [ ]:
import polars as pl

summary_path = common.result_path("feature_summary")
if os.path.exists(summary_path):
    summary = pl.read_csv(summary_path).filter(pl.col("space").is_in(common.SPACES))
    with pl.Config(tbl_rows=20, tbl_width_chars=160):
        print(summary)

    print("\nCohort sizes by space (alltime):")
    alltime = summary.filter(pl.col("window") == "alltime")
    if alltime.height:
        widest = alltime.get_column("n_patients").max()
        for row in alltime.sort("n_patients", descending=True).iter_rows(named=True):
            bar = "#" * int(40 * row["n_patients"] / widest)
            print(f"  {row['space']:10s} {row['n_patients']:7,d}  {bar}")
else:
    print(f"No summary at {summary_path} - has the run completed?")

## Next

`semantic_search/02_cluster.ipynb` clusters this representation;
`semantic_search/04_predict.ipynb` trains the supervised clinical-label models.